In [37]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import geopandas as gpd
import os
from dotenv import load_dotenv
from geoalchemy2.elements import WKBElement, WKTElement
load_dotenv()

True

### DATA BASE ENGINE

In [38]:
existing_database_url = os.getenv("BISELCO")
if existing_database_url:
    EXISTING_GIS_DATABASE= create_engine(existing_database_url)

current_database_url = os.getenv("BISELCOWEBSITE")
if current_database_url:
    CURRENT_GIS_DATABASE= create_engine(current_database_url)
    Session = sessionmaker(CURRENT_GIS_DATABASE)

In [39]:
# MUNICIPALITY DATA MIGRATION, NORMALIZATION AND PROCESSING
municipality = gpd.pd.read_sql(
    sql="""SELECT distinct municipality FROM gis.franchise_area order by municipality;""",
    con=EXISTING_GIS_DATABASE
)

values_mun = [(m,) for m in municipality['municipality'].to_list()]

# DATA MIGRATION FOR MUNICIPALITY VALUES
with Session.begin() as session:
    session.execute(statement=text(
        
        """INSERT INTO gis.municipality (name)
        VALUES (:mun)"""),
        params= [{"mun": m[0]} for m in values_mun] 
    )
    session.commit()
print("sucessfully inserted municipality")

sucessfully inserted municipality


In [40]:
# VILLAGE DATA MIGRATION, NORMALIZTION AND PROCESSING
village = gpd.pd.read_sql(
    sql = """SELECT distinct village, municipality FROM gis.franchise_area order by village;""",
    con=EXISTING_GIS_DATABASE
)

insert_stmt = text(
    """INSERT INTO gis.villages (municipality_id, name)
    VALUES (
        (SELECT id FROM gis.municipality WHERE name = :mun), :vill)"""
)
params = [{"mun": v[1], "vill": v[0]} for v in village.values]


with Session.begin() as session:
    session.execute(
        insert_stmt,
        params
    )
    session.commit()
print("sucessfully inserted village")

sucessfully inserted village


In [41]:
# BOUNDARY DATA MIGRATION, NORMALIZATION AND PROCESSING

boundary = gpd.read_postgis(
    sql = """SELECT distinct geom, municipality, village FROM gis.franchise_area;""",
    con=EXISTING_GIS_DATABASE,
    geom_col="geom"
)

params = [{"geom": b[0].wkt, "mun": b[1], "vill": b[2], "name": f"{b[2]} {b[1]}"} for b in boundary.values]

stmt = text(
    """INSERT INTO gis.boundary (geom, municipality_id, village_id, name)
    VALUES (:geom, (SELECT id FROM gis.municipality WHERE name = :mun), (SELECT id FROM gis.villages WHERE name = :vill), :name);"""
)

with Session.begin() as session:
    session.execute(stmt, params)
    session.commit()
print("sucessfully inserted boundary")

IntegrityError: (psycopg2.errors.NotNullViolation) null value in column "area" of relation "boundary" violates not-null constraint
DETAIL:  Failing row contains (1, 47, 4, 0103000020E6100000010000001D060000C84DE0B2B7ED5D404015233EBC8326..., PICAL LINAPACAN, null).

[SQL: INSERT INTO gis.boundary (geom, municipality_id, village_id, name)
    VALUES (%(geom)s, (SELECT id FROM gis.municipality WHERE name = %(mun)s), (SELECT id FROM gis.villages WHERE name = %(vill)s), %(name)s);]
[parameters: [{'geom': 'POLYGON ((119.71433708100005 11.257295553000063, 119.71463273200004 11.257295528000043, 119.7146960880001 11.257316235000076, 119.7147383250001 11.25 ... (60147 characters truncated) ... 19.71415758 11.257316279000065, 119.71421037400012 11.257305919000032, 119.71428428700005 11.257305914000028, 119.71433708100005 11.257295553000063))', 'mun': 'LINAPACAN', 'vill': 'PICAL', 'name': 'PICAL LINAPACAN'}, {'geom': 'POLYGON ((119.68193414000007 11.247527177000052, 119.68198671700009 11.247526061